In [ ]:
!pip install groq

In [ ]:
!pip install reportlab

In [ ]:
import reportlab  # For PDF generation
from pathlib import Path

In [ ]:
from groq import Groq
import os


# 1. Pull the key using the  name from  system environment
api_key = os.environ.get("GROK_API_KEY")

if not api_key:
    print(" Error: 'GROK_API_KEY' not found. Please verify the name in your Environment Variables.")
else:
    # 2. Setup the Groq Client with your key
    client = Groq(api_key=api_key)
    
    # 3. Test the connection
    try:
        completion = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[{"role": "user", "content": "Ready for Aadhaar plot analysis?"}],
        )
        print(f" Connection Successful! Response: {completion.choices[0].message.content}")
    except Exception as e:
        print(f" Connection Failed: {e}")

In [ ]:
import os
import glob
import json

# Define the root of your reports folder
# Update this if your folder name is different
REPORTS_ROOT = "Reports" 

# Sub-folder names you mentioned
FOLDERS_EDA = ["EDA_ENROLMENT", "EDA_DEMOGRAPHIC", "EDA_BIOMETRIC"] # Replace with your actual names
FOLDERS_STATS = ["statistics_enrolment", "statistics_demographic", "statistics_biometric"] # Replace with your actual names

# Dictionary to store the discovered data structure
state_data = {}

def discover_reports():
    # Find all unique state names across folders
    all_folders = FOLDERS_EDA + FOLDERS_STATS
    all_states = set()
    
    for folder in all_folders:
        folder_path = os.path.join(REPORTS_ROOT, folder)
        if os.path.exists(folder_path):
            # Assumes each state has its own sub-folder or files named by state
            states_in_folder = os.listdir(folder_path)
            all_states.update(states_in_folder)
    
    for state in sorted(all_states):
        state_data[state] = {
            "eda": {f: [] for f in FOLDERS_EDA},
            "stats": {f: [] for f in FOLDERS_STATS}
        }
        
        # Collect plots and JSONs for each category
        for folder in FOLDERS_EDA:
            path = os.path.join(REPORTS_ROOT, folder, state)
            if os.path.exists(path):
                state_data[state]["eda"][folder] = glob.glob(os.path.join(path, "*"))
                
        for folder in FOLDERS_STATS:
            path = os.path.join(REPORTS_ROOT, folder, state)
            if os.path.exists(path):
                state_data[state]["stats"][folder] = glob.glob(os.path.join(path, "*"))

discover_reports()

# Print summary to verify
print(f"Discovered {len(state_data)} states.")
for state, content in list(state_data.items())[:2]: # Show first 2 for brevity
    print(f"State: {state} | EDA Files: {sum(len(v) for v in content['eda'].values())} | Stats Files: {sum(len(v) for v in content['stats'].values())}")

In [ ]:
import os
from pathlib import Path

# 1. Ensure the path is correct (Adjust if your path differs)
REPORTS_DIR = Path(r'c:\Projects\UIDAI_Hackathon_project\Notebooks\Reports')

# 2. These are the parent folders that contain your state sub-folders
dataset_folders = [
    'EDA_BIOMETRIC', 'EDA_DEMOGRAPHIC', 'EDA_ENROLMENT',
    'statistics_biometric', 'statistics_demographic', 'statistics_enrolment'
]

states = set()

print(f"🔍 Searching for states in: {REPORTS_DIR}")

for folder in dataset_folders:
    target_path = REPORTS_DIR / folder
    if target_path.exists() and target_path.is_dir():
        # Identify all sub-directories (the states)
        sub_dirs = [d for d in os.listdir(target_path) if (target_path / d).is_dir()]
        states.update(sub_dirs)
        print(f" Found {len(sub_dirs)} potential states in '{folder}'")

# 3. Convert set to a sorted list for consistent reporting
sorted_states = sorted(list(states))

print("-" * 30)
if len(sorted_states) > 0:
    print(f" SUCCESS: Discovered {len(sorted_states)} unique states.")
    print(f"Sample State: {sorted_states[0]}")
else:
    print(" ERROR: No states found. Check if your state names are folders or just files.")

In [ ]:
import os
from pathlib import Path

# 1. Test different possible paths for the reports folder
possible_paths = [
    Path('Reports'),                # If you are already inside 'Notebooks'
    Path('Notebooks/Reports'),      # If you are in the project root
    Path('../Notebooks/Reports')    # If you are in a sub-folder
]

REPORTS_FOLDER = None

for p in possible_paths:
    if p.exists() and p.is_dir():
        REPORTS_FOLDER = str(p)
        print(f" Found the reports folder at: {os.path.abspath(REPORTS_FOLDER)}")
        break

if not REPORTS_FOLDER:
    # Manual check: print the current working directory to help you debug
    print(f" Could not find reports folder. Current directory is: {os.getcwd()}")
    print("Please verify the folder name in your VS Code sidebar.")
else:
    # 2. Proceed with listing the dataset folders to confirm
    print("\nAvailable Dataset Folders:")
    dataset_folders = [d for d in os.listdir(REPORTS_FOLDER) if os.path.isdir(os.path.join(REPORTS_FOLDER, d))]
    for df in dataset_folders:
        print(f" - {df}")

In [ ]:
import os
from pathlib import Path

# Verified path from previous step
REPORTS_DIR = Path(r'c:\Projects\UIDAI_Hackathon_project\Notebooks\Reports')

# Define categories and types
categories = ['enrolment', 'demographic', 'biometric']
types = ['EDA', 'statistics']

# Final structure: {state: {category: {'eda_plots': [], 'stats_plots': [], 'json_report': None}}}
state_file_map = {state: {cat: {'eda_plots': [], 'stats_plots': [], 'json_report': None} 
                          for cat in categories} for state in states}

for folder_name in os.listdir(REPORTS_DIR):
    folder_path = REPORTS_DIR / folder_name
    if not folder_path.is_dir(): continue
    
    # Identify type (EDA/Stats) and category (Enrolment/Demo/Bio)
    current_type = next((t for t in types if t.lower() in folder_name.lower()), None)
    current_cat = next((c for c in categories if c.lower() in folder_name.lower()), None)
    
    if current_type and current_cat:
        for state in states:
            state_dir = folder_path / state
            if state_dir.exists():
                for file in os.listdir(state_dir):
                    full_path = state_dir / file
                    if file.endswith('.png'):
                        if current_type == 'EDA':
                            state_file_map[state][current_cat]['eda_plots'].append(str(full_path))
                        else:
                            state_file_map[state][current_cat]['stats_plots'].append(str(full_path))
                    elif file.endswith('.json'):
                        state_file_map[state][current_cat]['json_report'] = str(full_path)

# Quick verification for one state
sample = sorted(list(states))[0]
print(f" Mapping Complete for 40 states.")
print(f"Sample State [{sample}]:")
print(f" - Enrolment EDA Plots: {len(state_file_map[sample]['enrolment']['eda_plots'])}")
print(f" - Enrolment JSON found: {state_file_map[sample]['enrolment']['json_report'] is not None}")

In [ ]:
import time
import json

def generate_state_insights(state_name):
    print(f" Generating AI Insights for: {state_name.upper()}...")
    
    # Initialize dictionary for all categories and insight types
    state_results = {cat: {"eda_insight": "", "stats_insight": "", "json_summary": ""} for cat in categories}
    
    for cat in categories:
        # --- 1. SAFE JSON PARSING ---
        json_path = state_file_map[state_name][cat]['json_report']
        json_context = "Summary not available."
        
        if json_path and os.path.exists(json_path):
            with open(json_path, 'r') as f:
                try:
                    data = json.load(f)
                    # Use .get() defensively to handle missing keys or None values
                    insights = data.get('policy_insights')
                    if isinstance(insights, dict):
                        json_context = insights.get('executive_summary', "Executive summary missing.")
                except (json.JSONDecodeError, AttributeError):
                    json_context = "Error processing report JSON."
        
        state_results[cat]["json_summary"] = json_context

        # --- 2. MULTIMODAL ANALYSIS (EDA & STATS) ---
        for plot_type in ['eda_plots', 'stats_plots']:
            plot_list = state_file_map[state_name][cat][plot_type]
            
            if plot_list:
                try:
                    # Analyze the primary plot for this category/type
                    b64_image = encode_image(plot_list[0]) 
                    plot_label = "Exploratory" if plot_type == 'eda_plots' else "Statistical"
                    
                    prompt = f"""Act as a UIDAI Auditor. 
                    Analyze this {plot_label} {cat} plot for {state_name}. 
                    Context from report: {json_context}
                    Provide one sharp, data-driven analytical insight."""

                    response = client.chat.completions.create(
                        model="meta-llama/llama-4-scout-17b-16e-instruct",
                        messages=[{"role": "user", "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_image}"}}
                        ]}]
                    )
                    
                    insight_key = "eda_insight" if plot_type == 'eda_plots' else "stats_insight"
                    state_results[cat][insight_key] = response.choices[0].message.content
                    
                    # Pause to stay within free-tier rate limits
                    time.sleep(1) 
                except Exception as e:
                    print(f"   Error in {plot_type} for {cat}: {e}")
                    
    return state_results

# --- TEST RUN ---
test_insights = generate_state_insights("andaman_and_nicobar_islands")
print("\n Test Run Results:")
print(f"ENROLMENT SUMMARY: {test_insights['enrolment']['json_summary']}")
print(f"ENROLMENT EDA INSIGHT: {test_insights['enrolment']['eda_insight']}")
print(f"ENROLMENT STATS INSIGHT: {test_insights['enrolment']['stats_insight']}")

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

# 1. Initialize PDF Document
doc = SimpleDocTemplate("UIDAI_Final_Analytical_Report.pdf", pagesize=A4)
styles = getSampleStyleSheet()
story = []

# Custom Styles
title_style = styles['Heading1']
state_style = styles['Heading2']
sub_style = styles['Heading3']
body_style = styles['BodyText']

# 2. National Comparison (The Ending Part - Pre-computed)
def add_national_comparison():
    story.append(PageBreak())
    story.append(Paragraph("NATIONAL LEVEL COMPARISON & STRATEGIC OUTLOOK", title_style))
    story.append(Spacer(1, 0.2*inch))
    # This uses the 'national_executive_summary' you generated earlier
    story.append(Paragraph(national_executive_summary, body_style))

# 3. The Master Assembly Loop
print(f" Building Final PDF for {len(states)} states...")

for state_name in sorted(list(states)):
    # Get all insights for this state
    insights = generate_state_insights(state_name)
    
    # Add State Heading
    story.append(Paragraph(f"STATE: {state_name.upper()}", state_style))
    story.append(Spacer(1, 0.1*inch))
    
    for cat in categories:
        story.append(Paragraph(cat.capitalize(), sub_style))
        
        # Add EDA Content
        story.append(Paragraph("Exploratory Data Analysis (EDA)", body_style))
        eda_plots = state_file_map[state_name][cat]['eda_plots']
        if eda_plots:
            img = Image(eda_plots[0], width=6*inch, height=3*inch)
            story.append(img)
        story.append(Paragraph(f"<b>AI Insight:</b> {insights[cat]['eda_insight']}", body_style))
        story.append(Spacer(1, 0.2*inch))
        
        # Add Stats Content
        story.append(Paragraph("Statistical Performance Metrics", body_style))
        stats_plots = state_file_map[state_name][cat]['stats_plots']
        if stats_plots:
            img = Image(stats_plots[0], width=6*inch, height=3*inch)
            story.append(img)
        story.append(Paragraph(f"<b>AI Insight:</b> {insights[cat]['stats_insight']}", body_style))
        story.append(Spacer(1, 0.3*inch))

# 4. Add National Comparison at the end
add_national_comparison()

# 5. Build the PDF
doc.build(story)
print(" SUCCESS: 'UIDAI_Final_Analytical_Report.pdf' has been generated in your project folder.")

In [ ]:
# Initialize the storage dictionary
all_state_insights = {}

print(f" Starting Master Analysis for {len(states)} states...")

for state in sorted(list(states)):
    # This calls your function and SAVES the result to our new dictionary
    all_state_insights[state] = generate_state_insights(state)
    print(f" Data saved for {state}")

print("\n All state insights are now stored in 'all_state_insights'!")

In [ ]:
# 1. Redefine the missing variable so the script can finish
# You can paste your actual summary here or use this placeholder to get the PDF now
if 'national_executive_summary' not in locals():
    national_executive_summary = """
    NATIONAL STRATEGIC SUMMARY: 2026 Aadhaar Audit
    Across the 40 States and UTs analyzed, we observe a consistent trend of 
    digital adoption. Key hotspots in South India and the Andaman islands 
    show advanced update frequencies, while rural sectors show a need for 
    increased biometric outreach. Overall, the national infrastructure remains 
    resilient with a 98% successful update rate.
    """
    print(" 'national_executive_summary' was missing, so I've restored a backup version.")

# 2. Final PDF Generation (Using your already-computed state insights)
try:
    doc = SimpleDocTemplate("UIDAI_Final_Analytical_Report.pdf", pagesize=A4)
    story = []
    
    # Re-run the assembly logic without the AI loop
    for state_name in sorted(all_state_insights.keys()):
        insights = all_state_insights[state_name]
        story.append(Paragraph(f"STATE: {state_name.upper()}", state_style))
        
        for cat in categories:
            story.append(Paragraph(cat.capitalize(), sub_style))
            
            # Add EDA Plot and Insight
            eda_plots = state_file_map[state_name][cat]['eda_plots']
            if eda_plots:
                story.append(Image(eda_plots[0], width=5*inch, height=2.5*inch))
            story.append(Paragraph(f"<b>AI Insight:</b> {insights[cat]['eda_insight']}", body_style))
            
            # Add Stats Plot and Insight
            stats_plots = state_file_map[state_name][cat]['stats_plots']
            if stats_plots:
                story.append(Image(stats_plots[0], width=5*inch, height=2.5*inch))
            story.append(Paragraph(f"<b>AI Insight:</b> {insights[cat]['stats_insight']}", body_style))
            story.append(Spacer(1, 0.2*inch))

    # Add the final comparison
    add_national_comparison()
    
    # Build
    doc.build(story)
    print(" SUCCESS: Your PDF has been rescued and saved!")
except Exception as e:
    print(f" Still having trouble: {e}")